# Bayesian Bug-Report Triage (BO) untuk Data Nyata -- Apache HBase (GitBugs) -- Notebook Paralel Penuh

## Notebook ini adalah TIRUAN STRUKTUR PENUH dari `Uji Coba Reduction.ipynb`

Setiap eksperimen (EXP0/Step4 sampai EXP10), setiap chart, setiap sel kode inti (kernel, GP, acquisition function, stopping rule, APFD, `run_reduction_loop`) **SAMA PERSIS** dengan notebook utama. Yang berbeda betul-betul cuma **datanya**: 250 bug report ASLI dari Apache HBase (proyek open source, lewat dataset publik [GitBugs](https://github.com/av9ash/gitbugs), Chandra, arXiv:2504.09651, CC BY 4.0), menggantikan 69 test case QA dummy Firebase Chat.

## Kenapa notebook ini dibuat
Notebook utama pakai oracle bug DUMMY yang sengaja dikonsentrasikan di 2 Menu (label buatan peneliti). Ini pertanyaan penting yang belum terjawab notebook utama sendiri: apakah mekanisme BO-nya (pilih item berikutnya berdasar kemiripan teks + ketidakpastian model, dibanding baca acak) tetap unggul kalau labelnya bug **SUNGGUHAN**, bukan buatan?

## 3 penyesuaian wajib karena datanya beda domain (SEMUA didokumentasikan di sini, bukan disembunyikan)

1. **Tidak ada kolom "Menu".** Bug report tidak punya struktur menu/fitur seperti test case QA. Sebagai gantinya dipakai kolom **`Priority`** (Blocker/Critical/Major/Minor/Trivial -- 5 nilai, tanpa data kosong) untuk: (a) warm start "1 item per kelompok" dan (b) EXP9's uji ketahanan "kelompok mana yang berisiko". **Keterbatasan yang diakui:** Menu di notebook utama punya puluhan nilai unik; Priority cuma 5, jadi warm start di sini jauh lebih kecil (5 item, ~2% dari 250) dibanding notebook utama (puluhan test case, ~25% dari 69), dan EXP9's "2 dari sekian kelompok berisiko" mencakup proporsi kelompok yang jauh lebih besar (2 dari 5 = 40%, vs 2 dari puluhan Menu).
2. **Tidak ada `cost_minutes` terukur.** Diganti proxy kasar: panjang teks laporan (`clip(panjang_teks/400, 1, 30)` menit) -- laporan lebih panjang diasumsikan makan waktu baca lebih lama. Ini heuristik eksplisit, persis seperti `cost_minutes` di notebook utama yang juga cuma estimasi manual, bukan waktu terukur sungguhan.
3. **Oracle bug-nya NYATA, bukan disuntikkan.** `is_severe_fixed_bug = 1` kalau `Priority` masuk Blocker/Critical **DAN** `Resolution == "Fixed"` -- field asli yang diisi maintainer HBase, bukan label buatan peneliti. Konsekuensinya: EXP1-EXP7 dan EXP10 di sini jalan langsung di atas oracle REAL ini (tidak ada `draw_clustered_oracle` seperti di notebook utama). EXP8 dan EXP9 tetap memakai fungsi penyuntik oracle SINTETIS yang sama persis dengan notebook utama (`draw_uniform_oracle`, `draw_clustered_oracle`) -- tapi maknanya sedikit bergeser di sini: bukan lagi "apakah hasil EXP1-7 cuma karena cara oracle dummy ditaruh", melainkan **"kalau oracle real ini kita ganti sementara dengan oracle sintetis, apakah polanya konsisten dengan notebook utama"**. Ini dijelaskan lagi di markdown EXP8/EXP9 masing-masing.

## Yang SAMA PERSIS dengan notebook utama (tidak diubah sedikit pun)
Kernel (cosine/RBF/Matern3-2/Matern5-2/RQ), GP Classification dengan Laplace approximation + probit link, 6 acquisition function (EI/LogEI/UCB/PI/Thompson/Cost-UCB), stopping rule berbasis theta, `compute_apfd` (Rothermel et al. 1999), `run_reduction_loop`, dan urutan+struktur 11 blok eksperimen (Step4/EXP0 kalibrasi theta, EXP1 vectorizer, EXP2 kernel, EXP3 HP tuning, EXP4 acquisition function, EXP5 beta, EXP6 theta sweep, EXP7 budget vs acak, EXP8 fairness check, EXP9/EXP9b robustness check, EXP10 pipeline gabungan) -- termasuk ke-9 vectorizer yang sama (TF-IDF, Feature Hashing, One-Hot Encoding, Word2Vec, GloVe, FastText, ELMo, Flair, Multilingual E5 Large Instruct). Mode `PAPER_VECTORISER_MODE=quick` (default divalidasi di sandbox, cuma TF-IDF+Feature Hashing) vs `=all` (9 vectorizer penuh, jalankan ini di Hakusan) bekerja identik dengan notebook utama.

## Satu bug tersembunyi yang ketemu & diperbaiki saat menyusun notebook ini
Sel demo visualisasi 3-langkah EXP10 di notebook utama (bagian PCA/surrogate model) ternyata masih memakai `best_so_far = mean(train_y)` (versi LAMA yang salah), padahal `run_reduction_loop` yang sebenarnya sudah diperbaiki jadi `max(train_y)`. Ini cuma memengaruhi gambar demo 3-langkah (bukan angka recall/APFD final manapun), tapi tetap inkonsistensi matematis nyata. **Sudah diperbaiki di notebook ini**, dan sebaiknya diperbaiki juga di notebook utama.


In [ ]:
import os

# === EXPERIMENT CONFIGURATION (identik dengan notebook utama Cell 1) ===
PAPER_VECTORISERS = (
    "TF-IDF", "Feature Hashing", "Word2Vec", "GloVe",
    "FastText", "ELMo", "Flair",
)
ADDITIONAL_VECTORISERS = ("One-Hot Encoding", "Multilingual E5 Large Instruct")
EXPERIMENT_METHODS = PAPER_VECTORISERS + ADDITIONAL_VECTORISERS
QUICK_VECTORISERS = ("TF-IDF", "Feature Hashing")

def select_vectorisers(selection: str) -> tuple[str, ...]:
    requested = [item.strip() for item in selection.split(",") if item.strip()]
    if not requested or requested == ["quick"]:
        return QUICK_VECTORISERS
    if requested == ["all"]:
        return EXPERIMENT_METHODS
    return tuple(name for name in EXPERIMENT_METHODS if name in requested)

VECTORISER_SELECTION = os.getenv("PAPER_VECTORISER_MODE", "quick")
SELECTED_VECTORISERS = select_vectorisers(VECTORISER_SELECTION)
print(f"Selected vectorisers ({len(SELECTED_VECTORISERS)} total):", ", ".join(SELECTED_VECTORISERS))

KERNEL_NAME = os.getenv("BO_KERNEL", "cosine")
HP_TUNING = os.getenv("BO_HP_TUNING", "fixed")
AF_NAME = os.getenv("BO_AF", "ei")
STOP_THETA = float(os.getenv("BO_STOP_THETA", "0.1"))  # placeholder -- dikalibrasi ulang di Step 4/EXP0 di bawah
SIGMA2 = float(os.getenv("BO_SIGMA2", "1.0"))
NOISE_ALPHA = float(os.getenv("BO_NOISE_ALPHA", "0.01"))
UCB_BETA = float(os.getenv("BO_UCB_BETA", "2.0"))
STOP_BETA = float(os.getenv("BO_STOP_BETA", "0.15"))
N_FAIRNESS_REPLICATES = int(os.getenv("BO_FAIRNESS_REPS", "10"))
N_CLUSTERED_REPLICATES = int(os.getenv("BO_CLUSTERED_REPS", "10"))
N_RISKY_GROUPS = int(os.getenv("BO_N_RISKY_GROUPS", "2"))  # analog N_RISKY_MENUS, lihat markdown intro


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

PERSISTENT_CACHE = Path.home() / ".mas_ai_cache"
PERSISTENT_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("GENSIM_DATA_DIR", str(PERSISTENT_CACHE / "gensim"))
os.environ.setdefault("TFHUB_CACHE_DIR", str(PERSISTENT_CACHE / "tfhub"))
os.environ.setdefault("HF_HOME", str(PERSISTENT_CACHE / "huggingface"))
os.environ.setdefault("SENTENCE_TRANSFORMERS_HOME", str(PERSISTENT_CACHE / "sentence_transformers"))
os.environ.setdefault("FLAIR_CACHE_DIR", str(PERSISTENT_CACHE / "flair"))
for _p in (PERSISTENT_CACHE / "gensim", PERSISTENT_CACHE / "tfhub", PERSISTENT_CACHE / "huggingface"):
    _p.mkdir(parents=True, exist_ok=True)

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

BASE_PACKAGES = {
    "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
    "seaborn": "seaborn", "pandas": "pandas", "numpy": "numpy",
}
for module_name, pip_name in BASE_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        install(pip_name)

if VECTORISER_SELECTION == "all":
    for pkg in ["gensim", "tensorflow", "tensorflow-hub", "flair>=0.14,<0.15",
                "sentence-transformers>=2.7,<3.0", "setuptools<70"]:
        install(pkg)
    print("Semua paket vectorizer terinstal.")
else:
    print("Mode quick: hanya TF-IDF dan Feature Hashing yang tersedia.")


In [ ]:
from pathlib import Path
import hashlib, json, math, re, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import ndtr
from scipy.optimize import minimize
warnings.filterwarnings("ignore")


## Step 1 -- Muat 250 bug report ASLI Apache HBase

Sampel 250 bug report (35 di antaranya `is_severe_fixed_bug=1`, ~14% -- sengaja di-oversample dari proporsi asli ~3.6% di 5.359 bug report HBase, supaya jumlah positifnya cukup untuk dievaluasi; ini pilihan desain yang sama jenisnya dengan oracle dummy 13-bug di notebook utama yang juga bukan proporsi natural) sudah disiapkan sebelumnya dengan seed tetap (42) supaya reproducible.

In [ ]:
def find_data_file(filename):
    curr = Path(".").resolve()
    search_roots = [curr] + list(curr.parents) + [
        Path(r"C:/Users/radit/Project/VisualStudioProject/Skripsi/MAS AI").resolve(),
        Path(r"D:/Kuliah/SKRIPSI/RESEARCH TEMA PROKSI/AUTOMATED TESTING ANDROID/Dokumen Suitmedia").resolve(),
        Path(r"D:/Kuliah/SKRIPSI/RESEARCH TEMA PROKSI/AUTOMATED TESTING ANDROID/Dokumen Kepake").resolve(),
    ]
    for root in search_roots:
        if root.exists():
            target = root / "data" / "gitbugs" / filename
            if target.exists():
                return target
            found = list(root.glob(f"**/data/gitbugs/{filename}"))
            if found:
                return found[0]
    raise FileNotFoundError(
        f"Tidak menemukan {filename}. Pastikan folder experiment/bayesian/data/gitbugs/ "
        "ada di sekitar notebook ini."
    )

def load_bug_cases(path):
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    # Proxy biaya baca (bukan waktu terukur, sama seperti cost_minutes di notebook
    # utama yang juga hanya estimasi): laporan lebih panjang -> asumsi lebih lama dibaca.
    df["cost_minutes"] = np.clip(df["text"].fillna("").str.len().to_numpy() / 400.0, 1.0, 30.0)
    return df

DATA_PATH = find_data_file("hbase_bugs_sample_for_bo.csv")
cases = load_bug_cases(DATA_PATH)
N_CASES = len(cases)
MAX_TC = N_CASES  # alias -- run_reduction_loop's signature (identical to main notebook) defaults to MAX_TC
assert N_CASES == 250, f"Expected 250 bug reports, got {N_CASES}"
print(f"Loaded {N_CASES} bug report asli dari {DATA_PATH.name}")
print(f"Estimasi total waktu baca (proxy): {cases['cost_minutes'].sum():.0f} menit")


In [ ]:
# Oracle bug: REAL, bukan disuntikkan seperti 13 dummy bug di notebook utama.
# is_severe_fixed_bug sudah dihitung saat menyiapkan sampel: Priority in
# {Blocker, Critical} DAN Resolution == "Fixed" -- field asli dari maintainer HBase.
oracle = cases["is_severe_fixed_bug"].to_numpy(dtype=float)
BUG_COUNT = int(oracle.sum())
REAL_BUG_INDICES = set(int(i) for i in np.where(oracle == 1.0)[0])
GROUP_COL = "Priority"  # analog "Menu" -- lihat markdown intro untuk kenapa & keterbatasannya

def draw_clustered_oracle(group_arr, case_count, bug_count, n_risky_groups, random_state):
    """SAMA PERSIS logikanya dengan draw_clustered_oracle di notebook utama
    (Cell 6 di sana), hanya digeneralisasi supaya menerima array kelompok apa
    pun (di notebook utama hardcoded ke cases_df['Menu']; di sini dipanggil
    dengan cases['Priority'] karena bug report tidak punya kolom Menu).
    Dipakai ulang oleh EXP9/EXP9b di bawah -- BUKAN dipakai untuk oracle utama
    EXP1-EXP7/EXP10, karena oracle utama di notebook ini sudah REAL."""
    rng_c = np.random.default_rng(random_state)
    unique_groups = sorted(set(group_arr))
    n_pick = min(n_risky_groups, len(unique_groups))
    risky_groups = list(rng_c.choice(unique_groups, size=n_pick, replace=False))
    candidate_idx = np.where(np.isin(group_arr, risky_groups))[0]
    if len(candidate_idx) < bug_count:
        remaining = np.setdiff1d(np.arange(case_count), candidate_idx)
        extra_needed = bug_count - len(candidate_idx)
        extra = rng_c.choice(remaining, size=extra_needed, replace=False)
        chosen = np.concatenate([candidate_idx, extra])
    else:
        chosen = rng_c.choice(candidate_idx, size=bug_count, replace=False)
    oracle_c = np.zeros(case_count)
    oracle_c[chosen] = 1.0
    return oracle_c, risky_groups

# Warm start: 1 item per nilai Priority unik (coverage-first) -- analog
# "1 TC per Menu" di notebook utama. TIDAK melihat label oracle sama sekali.
groups = cases[GROUP_COL].tolist()
seen_groups = set()
initial_indices = []
for i, g in enumerate(groups):
    if g not in seen_groups:
        seen_groups.add(g)
        initial_indices.append(i)

print(f"Oracle: {BUG_COUNT} bug serius+fixed ASLI dari {N_CASES} bug report ({BUG_COUNT/N_CASES:.1%})")
print(f"Warm start: {len(initial_indices)} item (1 per {GROUP_COL}) -- lebih kecil dari notebook "
      f"utama karena {GROUP_COL} cuma {cases[GROUP_COL].nunique()} nilai unik (vs puluhan Menu)")
print(f"Budget: maks {N_CASES} item, stopping theta={STOP_THETA} (placeholder, dikalibrasi di Step 4)\n")

print("=== WARM START INITIAL SEEDS (1 ITEM PER PRIORITY) ===")
warm_df = cases.iloc[initial_indices][["Issue id", "Priority", "Summary", "cost_minutes"]].copy()
warm_df["Summary"] = warm_df["Summary"].str.slice(0, 60)
warm_df["Bug serius+fixed ASLI?"] = oracle[initial_indices].astype(bool)
print(warm_df.to_string(index=False))

print(f"\n=== DISTRIBUSI BUG ASLI PER {GROUP_COL.upper()} ({BUG_COUNT} TOTAL) ===")
dist_df = cases.groupby(GROUP_COL)["is_severe_fixed_bug"].agg(["sum", "count"]).rename(
    columns={"sum": "bug_serius_fixed", "count": "total_item"})
dist_df["persen"] = (dist_df["bug_serius_fixed"] / dist_df["total_item"] * 100).round(1)
print(dist_df.to_string())


## Step 2 -- Ubah teks bug report jadi angka

Sama seperti notebook utama: bangun 9 matriks independen, satu per vectorizer, supaya EXP1 bisa membandingkannya secara adil. 6 metode berbasis embedding pretrained tetap gagal-jelas (`RuntimeError`) kalau paketnya belum terinstal, bukan diam-diam diganti angka acak. Mode `quick` (default divalidasi di sini) cuma memakai TF-IDF + Feature Hashing.

In [ ]:
TEXT_FIELDS = ["Summary", "Description"]  # analog Menu/Submenu/Skenario/dst -- lihat markdown intro
available_cols = [c for c in TEXT_FIELDS if c in cases.columns]
corpus = cases[available_cols].fillna("").apply(lambda r: " ".join(r.astype(str)), axis=1).tolist()

from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.preprocessing import normalize

def build_matrices(methods, corpus):
    matrices = {}
    for method in methods:
        if method == "TF-IDF":
            vec = TfidfVectorizer(max_features=500)
            X = vec.fit_transform(corpus).toarray()
        elif method == "Feature Hashing":
            vec = HashingVectorizer(n_features=256, alternate_sign=False)
            X = vec.transform(corpus).toarray()
        elif method == "One-Hot Encoding":
            tokens = [set(doc.lower().split()) for doc in corpus]
            vocab = sorted(set(w for t in tokens for w in t))
            X = np.array([[1.0 if w in t else 0.0 for w in vocab] for t in tokens])
        elif method == "Word2Vec":
            try:
                import gensim.downloader as api
                w2v = api.load("word2vec-google-news-300")
                X = np.array([np.mean([w2v[w] for w in doc.lower().split() if w in w2v] or [np.zeros(300)], axis=0) for doc in corpus])
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build Word2Vec embeddings: {exc}. Install gensim "
                    "(pip install gensim), or run with PAPER_VECTORISER_MODE=quick "
                    "to skip this vectoriser instead of silently substituting random noise."
                ) from exc
        elif method == "GloVe":
            try:
                import gensim.downloader as api
                glove = api.load("glove-wiki-gigaword-100")
                X = np.array([np.mean([glove[w] for w in doc.lower().split() if w in glove] or [np.zeros(100)], axis=0) for doc in corpus])
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build GloVe embeddings: {exc}. Install gensim "
                    "(pip install gensim), or run with PAPER_VECTORISER_MODE=quick "
                    "to skip this vectoriser instead of silently substituting random noise."
                ) from exc
        elif method == "FastText":
            try:
                import gensim.downloader as api
                ft = api.load("fasttext-wiki-news-subwords-300")
                X = np.array([np.mean([ft[w] for w in doc.lower().split() if w in ft] or [np.zeros(300)], axis=0) for doc in corpus])
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build FastText embeddings: {exc}. Install gensim "
                    "(pip install gensim), or run with PAPER_VECTORISER_MODE=quick "
                    "to skip this vectoriser instead of silently substituting random noise."
                ) from exc
        elif method == "ELMo":
            try:
                import tensorflow as tf
                import tensorflow_hub as hub
                elmo = hub.load("https://tfhub.dev/google/elmo/3")
                embed_fn = elmo.signatures["default"]
                input_names = list(embed_fn.structured_input_signature[1].keys())
                if not input_names:
                    raise RuntimeError("ELMo 'default' signature exposes no keyword inputs")
                X = embed_fn(**{input_names[0]: tf.constant(corpus)})["default"].numpy()
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build ELMo embeddings: {exc}. This TF1-format module "
                    "needs hub.load(...).signatures['default'](...) under modern "
                    "tensorflow_hub (hub.Module was removed), or run with "
                    "PAPER_VECTORISER_MODE=quick to skip this vectoriser instead of "
                    "silently substituting random noise."
                ) from exc
        elif method == "Flair":
            try:
                from flair.embeddings import WordEmbeddings, DocumentPoolEmbeddings
                from flair.data import Sentence
                document_embeddings = DocumentPoolEmbeddings([WordEmbeddings('glove')])
                sentences = [Sentence(doc) for doc in corpus]
                document_embeddings.embed(sentences)
                X = np.array([s.embedding.cpu().numpy() for s in sentences])
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build Flair embeddings: {exc}. Install flair, or run "
                    "with PAPER_VECTORISER_MODE=quick to skip this vectoriser instead "
                    "of silently substituting random noise."
                ) from exc
        elif method == "Multilingual E5 Large Instruct":
            try:
                from sentence_transformers import SentenceTransformer
                model = SentenceTransformer('intfloat/multilingual-e5-large')
                X = model.encode(corpus, show_progress_bar=False)
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build Multilingual E5 embeddings: {exc}. Install "
                    "sentence-transformers, or run with PAPER_VECTORISER_MODE=quick "
                    "to skip this vectoriser instead of silently substituting random noise."
                ) from exc
        else:
            X = np.random.RandomState(48).randn(len(corpus), 64)
        X = normalize(X, norm="l2")
        matrices[method] = X
        print(f"  {method:32s}: shape {X.shape}")
    return matrices

print(f"Building matrices for all selected methods: {SELECTED_VECTORISERS}")
matrices = build_matrices(SELECTED_VECTORISERS, corpus)


## Step 3 -- Model sesungguhnya: GPC dengan probit link, plus aturan berhenti

Bagian ini adalah inti notebook, sama persis dengan notebook utama.

**Kenapa bukan regresi biasa?** Hasil bug bersifat biner (ditemukan / tidak), bukan angka kontinu, jadi Gaussian Process biasa (dibuat untuk pengukuran kontinu yang berisik) bukan alat yang tepat. Gaussian Process **Classification** memperbaiki ini: setiap item dianggap punya "skor risiko" tersembunyi (`f`), dan fungsi *probit* (CDF normal standar, `Φ`) memampatkan skor itu jadi probabilitas 0-1 yang sah. Karena tidak ada rumus sederhana untuk skor tersembunyi yang tepat, model memakai **Laplace approximation** -- iterasi Newton (mengikuti Rasmussen & Williams, *Gaussian Processes for Machine Learning*, 2006, Algorithm 3.1) yang mencari skor tersembunyi paling mungkin dari hasil yang sudah teramati.

Detail penting: mengubah skor tersembunyi itu balik jadi probabilitas punya rumus **eksak** untuk model probit -- `Φ(mean / sqrt(1 + variance))` -- tanpa aproksimasi. Ini rumus yang sama yang sudah diperbaiki di notebook utama (versi lama salah pakai konstanta `π/8` dari model logistik), dan kode di bawah adalah salinan yang sudah diperbaiki itu.

**Cara memilih item berikutnya.** Begitu model punya probabilitas prediksi + ketidakpastian untuk setiap item tersisa, tiap item diberi skor lewat acquisition function -- default Expected Improvement (EI). EI dan lima alternatifnya (dicoba di EXP4) tidak mempertimbangkan biaya sama sekali -- dipilih sebagai default karena tujuan EXP1-EXP7 adalah menemukan bug sesedikit mungkin item yang dibaca, dan Cost-UCB (salah satu alternatif) justru menukar itu demi item yang murah-tapi-kurang-informatif. EXP4 membandingkan keenam aturan; EXP6 sengaja tetap pakai Cost-UCB untuk menunjukkan bentuk kurva berhenti yang sadar-biaya.

**Kapan berhenti lebih awal.** Setelah tiap pilihan, model cek: untuk semua item yang tersisa, apakah bahkan tebakan paling optimis (mean + margin aman) di bawah ambang kecil theta? Kalau ya untuk semuanya, berhenti -- inti dari "reduction", bukan cuma "reorder". `STOP_BETA` (margin aman untuk cek berhenti) sengaja dipisah dari `UCB_BETA` (untuk memilih item berikutnya), mengikuti literatur stopping-criterion (Wang et al. 2026, arXiv:2605.22561; Wilson et al. 2024, arXiv:2402.16811).


In [ ]:
import numpy as np
from scipy.special import ndtr
from scipy.optimize import minimize as sp_minimize

# ── Kernel functions ─────────────────────────────────────────────────────────────

def _cosine_kernel(X, Y):
    return X @ Y.T

def _rbf_kernel(X, Y, length_scale=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    return np.exp(-0.5 * np.sum(diff**2, axis=-1) / length_scale**2)

def _matern32_kernel(X, Y, length_scale=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    r = np.sqrt(np.sum(diff**2, axis=-1))
    v = r / length_scale
    return (1 + np.sqrt(3)*v) * np.exp(-np.sqrt(3)*v)

def _matern52_kernel(X, Y, length_scale=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    r = np.sqrt(np.sum(diff**2, axis=-1))
    v = r / length_scale
    return (1 + np.sqrt(5)*v + 5*v**2/3) * np.exp(-np.sqrt(5)*v)

def _rq_kernel(X, Y, length_scale=1.0, alpha=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    r2 = np.sum(diff**2, axis=-1)
    return (1 + r2 / (2*alpha*length_scale**2))**(-alpha)

def get_kernel(name, X, Y, sigma2=1.0, length_scale=1.0):
    if name == "cosine":
        return sigma2 * _cosine_kernel(X, Y)
    elif name == "rbf":
        return sigma2 * _rbf_kernel(X, Y, length_scale)
    elif name == "matern32":
        return sigma2 * _matern32_kernel(X, Y, length_scale)
    elif name == "matern52":
        return sigma2 * _matern52_kernel(X, Y, length_scale)
    elif name == "rq":
        return sigma2 * _rq_kernel(X, Y, length_scale)
    else:
        raise ValueError(f"Unknown kernel: {name}")

# ── Probit derivatives (Laplace GPC) ────────────────────────────────────────────

def _probit_derivatives(f, y):
    yf = y * f
    phi = ndtr(yf)
    phi = np.clip(phi, 1e-10, 1 - 1e-10)
    pdf = np.exp(-0.5 * yf**2) / np.sqrt(2 * np.pi)
    grad = y * pdf / phi
    W = (pdf / phi)**2 + yf * pdf / phi
    return grad, W

# ── MLE for kernel hyperparameters (EXP3) ───────────────────────────────────────

def _neg_lml(log_params, X, y, kernel_name, noise):
    sigma2 = np.exp(log_params[0])
    length_scale = np.exp(log_params[1]) if len(log_params) > 1 else 1.0
    n = len(y)
    K = get_kernel(kernel_name, X, X, sigma2, length_scale)
    K += noise * np.eye(n)
    try:
        L = np.linalg.cholesky(K)
        alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
        lml = -0.5 * y @ alpha - np.sum(np.log(np.diag(L))) - 0.5 * n * np.log(2*np.pi)
        return -lml
    except np.linalg.LinAlgError:
        return 1e10

def tune_hyperparams(X, y, kernel_name, noise, n_restarts=5):
    best_val, best_params = np.inf, [0.0, 0.0]
    for _ in range(n_restarts):
        x0 = np.random.randn(2)
        res = sp_minimize(_neg_lml, x0, args=(X, y, kernel_name, noise),
                          method="L-BFGS-B")
        if res.fun < best_val:
            best_val = res.fun
            best_params = res.x
    return np.exp(best_params[0]), np.exp(best_params[1])

# ── GPC predict (Laplace approximation) ──────────────────────────────────────────

def _gp_predict(train_x, train_y, cand_x,
                sigma2=SIGMA2, noise=NOISE_ALPHA,
                kernel_name=KERNEL_NAME, hp_tuning=HP_TUNING):
    if hp_tuning == "mle" and len(train_y) >= 5:
        sigma2, length_scale = tune_hyperparams(train_x, train_y*2-1, kernel_name, noise)
    else:
        length_scale = 1.0

    train_y_pm = train_y * 2 - 1  # {0,1} -> {-1,+1}
    n = len(train_y)
    K = get_kernel(kernel_name, train_x, train_x, sigma2, length_scale)
    K += noise * np.eye(n)

    # Laplace: Newton iterations
    f = np.zeros(n)
    for _ in range(20):
        grad, W = _probit_derivatives(f, train_y_pm)
        W_safe = np.maximum(W, 1e-8)
        W_sqrt = np.sqrt(W_safe)
        B = np.eye(n) + W_sqrt[:, None] * K * W_sqrt[None, :]
        L = np.linalg.cholesky(B + 1e-8 * np.eye(n))
        b = W_safe * f + grad
        v = np.linalg.solve(L, W_sqrt * (K @ b))
        f_new = K @ (b - W_sqrt * np.linalg.solve(L.T, v))
        if np.max(np.abs(f_new - f)) < 1e-6:
            f = f_new
            break
        f = f_new

    grad, W = _probit_derivatives(f, train_y_pm)
    W_safe = np.maximum(W, 1e-8)
    W_sqrt = np.sqrt(W_safe)
    B = np.eye(n) + W_sqrt[:, None] * K * W_sqrt[None, :]
    L = np.linalg.cholesky(B + 1e-8 * np.eye(n))

    K_star = get_kernel(kernel_name, cand_x, train_x, sigma2, length_scale)
    mu = K_star @ grad
    v = np.linalg.solve(L, W_sqrt[:, None] * K_star.T)
    K_ss = np.diag(get_kernel(kernel_name, cand_x, cand_x, sigma2, length_scale))
    var = np.maximum(K_ss - np.sum(v**2, axis=0), 1e-8)
    std = np.sqrt(var)

    # Squash to [0,1] probability
    kappa = 1.0 / np.sqrt(1 + var)  # FIXED: exact probit squashing (GPML eq 3.25);
    # the previous np.pi/8 constant is the MacKay logistic-approximation-to-probit
    # formula, wrong for a model whose own likelihood already IS probit.
    mean_prob = ndtr(kappa * mu)
    phi_star = np.exp(-0.5 * (kappa * mu)**2) / np.sqrt(2 * np.pi)
    std_prob = np.maximum(phi_star * kappa * std, 1e-6)
    return mean_prob, std_prob

# ── Acquisition functions ─────────────────────────────────────────────────────────

def _cost_ucb(mean, std, cost, beta=UCB_BETA):
    return (mean + beta * std) / np.sqrt(np.maximum(cost, 1.0))

def _ei(mean, std, best, xi=0.01):
    z = (mean - best - xi) / np.maximum(std, 1e-8)
    return (mean - best - xi) * ndtr(z) + std * np.exp(-0.5*z**2)/np.sqrt(2*np.pi)

def _logei(mean, std, best, xi=0.01):
    ei = _ei(mean, std, best, xi)
    return np.log(np.maximum(ei, 1e-30))

def _ucb(mean, std, beta=UCB_BETA):
    return mean + beta * std

def _pi(mean, std, best, xi=0.01):
    return ndtr((mean - best - xi) / np.maximum(std, 1e-8))

def _thompson(mean, std, rng):
    return rng.normal(mean, std)

def get_af_score(af_name, mean, std, cost, best, rng, beta=UCB_BETA):
    # FIXED: beta used to be silently dropped here -- _cost_ucb/_ucb fell back
    # to their own default parameter (bound to the global UCB_BETA at
    # definition time), so a caller passing a different beta into
    # run_reduction_loop (see EXP5) never actually changed what the
    # acquisition function did. Forwarding it here is what makes EXP5's
    # beta sweep real instead of a no-op.
    if af_name == "cost_ucb":
        return _cost_ucb(mean, std, cost, beta=beta)
    elif af_name == "ei":
        return _ei(mean, std, best)
    elif af_name == "logei":
        return _logei(mean, std, best)
    elif af_name == "ucb":
        return _ucb(mean, std, beta=beta)
    elif af_name == "pi":
        return _pi(mean, std, best)
    elif af_name == "ts":
        return _thompson(mean, std, rng)
    else:
        raise ValueError(f"Unknown AF: {af_name}")

# ── Stopping criterion ─────────────────────────────────────────────────────────────

def _should_stop(mean, std, stop_beta, theta):
    """Return True jika semua kandidat tersisa punya upper bound < theta.

    FIXED (grounded in the stopping-criterion literature -- Wang et al. 2026,
    "Regret-Based (eps,delta)-optimal Stopping Criteria for Bayesian
    Optimization", arXiv:2605.22561, and Wilson et al. 2024, "Stopping
    Bayesian Optimization with Probabilistic Regret Bounds",
    arXiv:2402.16811): two changes vs. the original version.
    (1) stop_beta is now a separate, decoupled confidence multiplier from
    the beta used by the UCB acquisition function -- both papers keep the
    acquisition function's beta_t untouched and use a tighter, independent
    parameter for the stop check itself. Wilson et al. specifically show
    that reusing the SAME bound for both (their "Delta-CB" baseline) is
    "unreliable" or needs post-hoc tuning -- exactly the symptom seen here
    (theta=0.05-0.3 never triggered a single stop with the shared beta=2.0).
    (2) the bound is now clipped to [0,1]: mean/std live on a probability
    scale (post probit-squash) but an unclipped mean+beta*std can exceed 1,
    which made it incomparable to a small probability threshold theta.
    """
    upper_bounds = np.clip(mean + stop_beta * std, 0.0, 1.0)
    return bool(np.all(upper_bounds < theta))

# ── APFD: Average Percentage of Faults Detected ─────────────────────────────────
#
# WHY THIS WAS ADDED: EXP1, EXP2, EXP3, EXP4, EXP5 and EXP7 all share the same
# default STOP_THETA=0.1, and EXP6 (the only experiment that actually varies
# theta) shows the stopping rule never triggers below roughly theta=0.5 on
# this dataset. That means every one of those other experiments silently runs
# to the fixed MAX_TC budget every single time, so the only thing being
# compared is "final bug_recall after the whole budget is spent" -- a metric
# that is blind to HOW FAST each option got there. Several settings (e.g.
# EI/LogEI/UCB/PI in EXP4, or rbf/matern32/matern52/rq in EXP2) end up
# selecting the same final SET of test cases once given the full budget, so
# they tie exactly on bug_recall even though they clearly take different
# paths to get there (verified by tracing per-step choices).
#
# APFD (Rothermel et al., "Test Case Prioritization: An Empirical Study",
# ICSM 1999 -- the standard metric in test-case-prioritization research for
# exactly this problem) scores HOW EARLY each fault is found, not just
# whether it is found by the end:
#     APFD = 1 - (TF_1 + TF_2 + ... + TF_m) / (n * m) + 1 / (2n)
# where n = number of test cases run, m = number of known faults, and TF_i
# is the (1-indexed) position of the first test case that exposes fault i.
#
# EXTENSION for a REDUCED suite: the original formula assumes every fault is
# eventually found somewhere in the n tests run. That assumption does not
# always hold here (this notebook deliberately stops before all 69 TC), so a
# fault never found within the executed subset is assigned TF_i = n + 1 --
# the standard worst-case convention (equivalent to "found the instant after
# the budget ran out"), so a missed bug always scores worse than one found on
# the very last test case, instead of silently dropping out of the average.
#
# UPDATE (Step 4, below): STOP_THETA=0.1 was the root cause of the tie
# described above, so it's no longer just a fallback -- Step 4 recalibrates
# it to a real, tested value before EXP1 runs. APFD stays regardless,
# because it answers a different question than recall does (how early vs.
# whether at all) and both are useful once theta actually varies things.
def compute_apfd(selected, oracle, total_bugs=None):
    n = len(selected)
    m = int(oracle.sum()) if total_bugs is None else total_bugs
    if n == 0 or m == 0:
        return 0.0
    pos = {tc_idx: step + 1 for step, tc_idx in enumerate(selected)}  # 1-indexed
    bug_indices = [i for i in range(len(oracle)) if oracle[i] == 1.0]
    tf = [pos.get(bi, n + 1) for bi in bug_indices]
    return 1 - (sum(tf) / (n * m)) + 1 / (2 * n)


# ── Reduction loop (CORE PERBEDAAN dari prioritization) ────────────────────────────

def run_reduction_loop(matrix, ids, menus, costs, oracle,
                       initial_indices,
                       max_tc=MAX_TC,
                       stop_theta=STOP_THETA,
                       kernel_name=KERNEL_NAME,
                       af_name=AF_NAME,
                       hp_tuning=HP_TUNING,
                       sigma2=SIGMA2,
                       noise=NOISE_ALPHA,
                       beta=UCB_BETA,
                       stop_beta=STOP_BETA,
                       rng=None):
    if rng is None:
        rng = np.random.default_rng(42)

    n = len(ids)
    selected = list(initial_indices)
    observed_oracle = {i: oracle[i] for i in selected}
    total_bugs = int(oracle.sum())

    history = []  # list of dicts per iteration

    for step in range(max_tc - len(initial_indices)):
        candidates = [i for i in range(n) if i not in set(selected)]
        if not candidates:
            break

        train_x = matrix[selected]
        train_y = np.array([observed_oracle[i] for i in selected])
        cand_x = matrix[candidates]
        cand_costs = np.array([costs[i] for i in candidates])

        mean, std = _gp_predict(train_x, train_y, cand_x,
                                sigma2=sigma2, noise=noise,
                                kernel_name=kernel_name, hp_tuning=hp_tuning)

        # Stopping check (stop_beta is deliberately separate from the
        # acquisition beta -- see _should_stop's docstring)
        if _should_stop(mean, std, stop_beta, stop_theta):
            break

        # AF selection
        # FIXED (2026-08-25): was np.mean(train_y), should be np.max(train_y).
        # train_y is a binary {0,1} bug-found label per already-run TC, and EI/
        # PI are defined relative to the BEST objective value achieved so far --
        # not the average. Using the mean turned "best" into a slowly-drifting
        # fraction (the running bug-rate among tests run), so EI/PI were scoring
        # improvement against the wrong reference the moment more than one TC
        # had been run. max(train_y) is 1.0 the instant any bug has been found
        # (0.0 before that), which is what "best objective value observed"
        # actually means for a binary target -- exactly what the classic EI/PI
        # formulas (Mockus 1978; Kushner 1964) assume "best" to be.
        best_so_far = float(np.max(train_y)) if len(train_y) > 0 else 0.0
        scores = get_af_score(af_name, mean, std, cand_costs, best_so_far, rng, beta=beta)
        chosen_local = int(np.argmax(scores))
        chosen = candidates[chosen_local]

        selected.append(chosen)
        observed_oracle[chosen] = oracle[chosen]

        bugs_found = sum(observed_oracle[i] for i in selected)
        recall = bugs_found / total_bugs if total_bugs > 0 else 0.0
        total_cost = sum(costs[i] for i in selected)

        history.append({
            "step": len(selected),
            "chosen_id": ids[chosen],
            "bugs_found": bugs_found,
            "bug_recall": recall,
            "total_cost_min": total_cost,
            "stopped_early": False
        })

    # Mark if stopped by theta (not by max_tc)
    stopped_by_theta = len(selected) < max_tc and len(selected) < n
    bugs_found = sum(observed_oracle[i] for i in selected)
    recall = bugs_found / total_bugs if total_bugs > 0 else 0.0

    apfd = compute_apfd(selected, oracle, total_bugs=total_bugs)

    return {
        "selected_indices": selected,
        "n_selected": len(selected),
        "bugs_found": bugs_found,
        "bug_recall": recall,
        "apfd": apfd,
        "stopped_by_theta": stopped_by_theta,
        "total_cost_min": sum(costs[i] for i in selected),
        "history": history
    }

print("GPC + Reduction loop defined.")
print(f"  kernel={KERNEL_NAME}, af={AF_NAME}, hp={HP_TUNING}")
print(f"  stop_theta={STOP_THETA}, max_tc={MAX_TC}")


## Step 4 -- Kalibrasi ambang berhenti SEBELUM menjalankan EXP1-EXP7

Sama seperti notebook utama: kalibrasi dulu supaya EXP1-EXP7 tidak diam-diam selalu membaca seluruh {N_CASES} item. **Catatan kejujuran metodologis:** grid theta di bawah (`0.10` s.d. `0.55`) sengaja TIDAK dikalibrasi ulang untuk skala 250-item -- dipakai persis sama seperti notebook utama (yang dikalibrasi untuk 69 item), supaya bisa dilihat apakah bentuk kurvanya konsisten atau butuh grid berbeda pada skala lebih besar. Ini bisa saja membuat kalibrasi di sini kurang optimal; itu temuan yang jujur dilaporkan apa adanya, bukan diam-diam diperbaiki.

In [ ]:

# ── Shared plotting helpers: same visual language as the PPT deck ─────────
# Bar height = test cases needed (the metric that actually differs once
# several options tie on recall -- plain recall-% bars used to look
# identical whenever multiple options hit 100%, so there was no visual way
# to tell which one to pick). Failing options are orange + hatched instead
# of just "shorter", so a short bar never reads as "better". The winning
# option gets an explicit "Best pick" badge. Real x/y axes with gridlines
# throughout -- no more hidden/blank axes.
_BLUE, _BLUE_DARK, _ORANGE, _GRAY = "#2A78D6", "#123E7A", "#EB6834", "#C3C2B7"
_TITLE_C, _SUB_C, _AXIS_C = "#0B0B0B", "#52514E", "#8A8983"

def plot_tc_bars(labels, tc_values, recall_values, xlabel, title, subtitle,
                  winner_idx=None, fail_threshold=0.95, savepath=None, figsize=(9, 5), ylabel="Test cases needed", unit="TC"):
    tc_values = [float(v) for v in tc_values]
    recall_values = [float(v) for v in recall_values]
    if winner_idx is None:
        passing = [i for i, r in enumerate(recall_values) if r >= 0.999]
        if passing:
            m = min(tc_values[i] for i in passing)
            winner_idx = next(i for i in passing if tc_values[i] == m)

    colors, hatches, edgecolors, lws = [], [], [], []
    for i, r in enumerate(recall_values):
        if r < fail_threshold:
            colors.append(_ORANGE); hatches.append("////"); edgecolors.append("white"); lws.append(0)
        elif i == winner_idx:
            colors.append(_BLUE); hatches.append(None); edgecolors.append(_BLUE_DARK); lws.append(2.2)
        else:
            colors.append(_GRAY); hatches.append(None); edgecolors.append("white"); lws.append(0)

    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(labels))
    bars = ax.bar(x, tc_values, color=colors, width=0.6, edgecolor=edgecolors, linewidth=lws, zorder=3)
    for bar, h in zip(bars, hatches):
        if h:
            bar.set_hatch(h)

    ymax = max(tc_values) * 1.4
    for xi, tc, r in zip(x, tc_values, recall_values):
        label = f"{int(round(tc))} {unit}\n{r*100:.0f}% found" + ("" if r >= fail_threshold else ", stopped early")
        ax.text(xi, tc + ymax * 0.03, label, ha="center", va="bottom", fontsize=10, color=_TITLE_C)
    if winner_idx is not None:
        ax.text(x[winner_idx], tc_values[winner_idx] + ymax * 0.19, "Best pick",
                ha="center", va="bottom", fontsize=10.5, color=_BLUE_DARK, fontweight="bold")

    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
    ax.set_xlabel(xlabel, fontsize=11, color=_SUB_C)
    ax.set_ylabel(ylabel, fontsize=11, color=_SUB_C)
    ax.set_ylim(0, ymax)
    ax.yaxis.set_major_locator(plt.MaxNLocator(5, integer=True))
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(_AXIS_C)
    ax.tick_params(colors=_AXIS_C)
    ax.yaxis.grid(True, color="#E7E6E1", linewidth=1)
    ax.set_axisbelow(True)
    fig.suptitle(title, x=0.01, ha="left", fontsize=14, fontweight="bold", color=_TITLE_C, y=0.99)
    ax.set_title(subtitle, loc="left", fontsize=10, color=_SUB_C, pad=10)
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    if savepath:
        os.makedirs("results", exist_ok=True)
        plt.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.show()
    return winner_idx


def plot_pct_sweep(x_labels, series, xlabel, title, subtitle, chosen_label=None, savepath=None, figsize=(9, 5)):
    """series: dict {name: (values_0to1, color)}. One 0-100% axis for every
    line -- no dual-axis (two different scales on one plot reads as
    confusing/misleading, so this notebook no longer uses twinx() for
    comparison charts)."""
    fig, ax = plt.subplots(figsize=figsize)
    x = list(range(len(x_labels)))
    for name, (vals, color) in series.items():
        ax.plot(x, [v * 100 for v in vals], "o-", color=color, linewidth=2.5, markersize=7, label=name, zorder=3)
        for xi, v in zip(x, vals):
            ax.annotate(f"{v*100:.0f}%", (xi, v * 100), textcoords="offset points",
                        xytext=(0, 8), ha="center", fontsize=9, color=_TITLE_C)
    if chosen_label is not None and chosen_label in x_labels:
        ci = x_labels.index(chosen_label)
        ax.axvline(ci, color="#B9B7AE", linestyle=(0, (4, 3)), linewidth=1.6, zorder=0)
        ax.text(ci, 112, "chosen", ha="center", fontsize=9, color=_SUB_C)
    ax.set_xticks(x); ax.set_xticklabels(x_labels, fontsize=10)
    ax.set_xlabel(xlabel, fontsize=11, color=_SUB_C)
    ax.set_ylim(0, 122)
    ax.set_yticks([0, 20, 40, 60, 80, 100]); ax.set_yticklabels([f"{v}%" for v in [0, 20, 40, 60, 80, 100]])
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(_AXIS_C)
    ax.tick_params(colors=_AXIS_C)
    ax.yaxis.grid(True, color="#E7E6E1", linewidth=1)
    ax.set_axisbelow(True)
    ax.legend(loc="center right", frameon=False, fontsize=9.5, labelcolor=_SUB_C)
    fig.suptitle(title, x=0.01, ha="left", fontsize=14, fontweight="bold", color=_TITLE_C, y=0.99)
    ax.set_title(subtitle, loc="left", fontsize=10, color=_SUB_C, pad=10)
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    if savepath:
        os.makedirs("results", exist_ok=True)
        plt.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.show()


# Setup referensi kalibrasi -- sama persis konvensinya dengan notebook utama
# (vectorizer pertama, kernel & AF default) karena EXP1/EXP2/EXP4 belum jalan
# di titik ini jadi belum ada "pemenang" untuk dikalibrasi.
costs = cases["cost_minutes"].values
ids = cases["Issue id"].tolist()

_calib_method = list(matrices.keys())[0]
_calib_matrix = matrices[_calib_method]

_theta_grid = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.52, 0.53, 0.55]
_calib_rows = []
for _theta in _theta_grid:
    _r = run_reduction_loop(
        matrix=_calib_matrix, ids=ids, menus=groups, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=N_CASES, stop_theta=_theta,
        kernel_name=KERNEL_NAME, af_name=AF_NAME, rng=np.random.default_rng(42)
    )
    _calib_rows.append({"theta": _theta, "TC_run": _r["n_selected"],
                         "bug_recall": _r["bug_recall"], "bugs_found": _r["bugs_found"],
                         "stopped_by_theta": _r["stopped_by_theta"]})
    print(f"  theta={_theta:.2f}: {_r['n_selected']:3d} item, recall={_r['bug_recall']:.1%}, stopped={_r['stopped_by_theta']}")

_df_calib = pd.DataFrame(_calib_rows)
_ceiling_recall = _df_calib["bug_recall"].max()
_lossless = _df_calib[_df_calib["bug_recall"] >= _ceiling_recall - 1e-9]
_CALIBRATED_THETA = float(_lossless["theta"].max())

print(f"\nRecall ceiling (theta praktis tak terbatas): {_ceiling_recall:.1%}")
print(f"Theta terbesar yang masih mencapai ceiling itu (zero bug loss): {_CALIBRATED_THETA}")

STOP_THETA = _CALIBRATED_THETA
os.makedirs("results", exist_ok=True)
_df_calib.to_csv("results/real_exp0_theta_calibration.csv", index=False)
print(f"\nSTOP_THETA sekarang {STOP_THETA} untuk semua eksperimen di bawah kecuali EXP6.")

_theta_labels = [f"{t:.2f}" for t in _df_calib["theta"]]
plot_pct_sweep(
    _theta_labels,
    {
        f"Item dibaca (% dari {N_CASES})": (_df_calib["TC_run"] / N_CASES, _BLUE),
        "Bug ditemukan (%)": (_df_calib["bug_recall"], _ORANGE),
    },
    xlabel="Keyakinan yang dibutuhkan sebelum berhenti (theta)",
    title="Seberapa yakin model harus berhenti lebih awal? (data bug ASLI)",
    subtitle=f"Di {STOP_THETA:.2f} sudah lewati item tanpa kehilangan bug; setelah itu mulai menukar bug demi berhenti lebih cepat",
    chosen_label=f"{STOP_THETA:.2f}",
    savepath="results/real_exp0_theta_calibration.png",
)


## EXP1 -- Representasi teks mana yang paling ampuh? (RQ1)

Sama seperti notebook utama: bandingkan (hingga) 9 cara membaca teks bug report, dengan kernel cosine + EI + theta terkalibrasi, tanpa batas budget selain aturan berhenti.

In [ ]:
reduction_results = {}
print(f"=== EXP1: MENJALANKAN REDUCTION LOOP UNTUK {len(matrices)} VECTORIZER (oracle REAL) ===\n")
for method_name, matrix in matrices.items():
    result = run_reduction_loop(
        matrix=matrix, ids=ids, menus=groups, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=N_CASES, stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME, af_name=AF_NAME, hp_tuning=HP_TUNING,
        rng=np.random.default_rng(42)
    )
    reduction_results[method_name] = result
    print(f"[{method_name:32s}] Dibaca {int(result['n_selected']):3d} item | "
          f"Bug ditemukan: {int(result['bugs_found']):2d}/{BUG_COUNT} ({result['bug_recall']:.1%}) | "
          f"APFD: {result['apfd']:.1%} | Stopped Early: {result['stopped_by_theta']}")
    print(f"  5 item pertama yang dipilih: {[h['chosen_id'] for h in result['history'][:5]]}...")

_ranked = sorted(
    reduction_results.items(),
    key=lambda kv: (-kv[1]["bug_recall"], kv[1]["n_selected"], -kv[1]["apfd"])
)
BEST_METHOD = _ranked[0][0]
print(f"\nEXP1 winner (dipakai sebagai 'best_method' oleh eksperimen di bawah, termasuk EXP10): {BEST_METHOD} "
      f"(recall={reduction_results[BEST_METHOD]['bug_recall']:.1%}, "
      f"item={reduction_results[BEST_METHOD]['n_selected']}, "
      f"APFD={reduction_results[BEST_METHOD]['apfd']:.1%})")


## Hasil EXP1, dalam tabel dan chart

In [ ]:
summary_rows = []
for method, r in reduction_results.items():
    summary_rows.append({
        "Method": method,
        "Item Dibaca": r["n_selected"],
        "Item Dibaca (%)": f"{r['n_selected']/N_CASES*100:.1f}%",
        "Bug Recall": f"{r['bug_recall']:.1%}",
        "APFD": f"{r['apfd']:.1%}",
        "Bugs Found": f"{r['bugs_found']}/{BUG_COUNT}",
        "Stopped by theta": r["stopped_by_theta"],
    })
df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))
os.makedirs("results", exist_ok=True)
df_summary.to_csv("results/real_exp1_vectorizer_summary.csv", index=False)
with open('results/real_exp1_reduction_results.json', 'w') as f:
    json.dump({k: {rk: (rv.tolist() if isinstance(rv, np.ndarray) else bool(rv) if isinstance(rv, (bool, np.bool_)) else int(rv) if isinstance(rv, (int, np.integer)) else float(rv) if isinstance(rv, (float, np.floating)) else rv) for rk, rv in v.items()} for k, v in reduction_results.items()}, f, indent=2)

methods = list(reduction_results.keys())
recalls = [reduction_results[m]["bug_recall"] for m in methods]
n_selected = [reduction_results[m]["n_selected"] for m in methods]

plot_tc_bars(
    methods, n_selected, recalls,
    xlabel="Cara membaca teks bug report (vectorizer)",
    title="Data bug ASLI (HBase): vectorizer mana yang temukan bug tercepat?",
    subtitle=f"Bar = jumlah bug report dibaca sebelum berhenti (theta={STOP_THETA:.2f}, maks {N_CASES}). Oranye = bug serius+fixed ASLI yang terlewat",
    winner_idx=methods.index(BEST_METHOD),
    savepath="results/real_exp1_vectorizer.png",
    figsize=(10, 5.5),
    ylabel="Bug report dibaca",
    unit="item",
)
print("EXP1 CSV & JSON tersimpan di results/")


## EXP2 -- Apakah fungsi kemiripan (kernel) berpengaruh? (RQ2)

Cosine vs RBF/Matern3-2/Matern5-2/RQ, vectorizer dikunci ke pemenang EXP1.

In [ ]:
KERNEL_OPTIONS = ['cosine', 'rbf', 'matern32', 'matern52', 'rq']
best_method = BEST_METHOD
best_matrix = matrices[best_method]
print(f"EXP2: Kernel sweep pakai vectorizer '{best_method}'")

kernel_results = []
for k_name in KERNEL_OPTIONS:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=groups, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=N_CASES, stop_theta=STOP_THETA,
        kernel_name=k_name, af_name=AF_NAME, hp_tuning=HP_TUNING, rng=np.random.default_rng(42)
    )
    kernel_results.append({"kernel": k_name, "TC_run": r["n_selected"], "bug_recall": r["bug_recall"],
                            "apfd": r["apfd"], "bugs_found": r["bugs_found"], "stopped_by_theta": r["stopped_by_theta"]})
    print(f"  kernel={k_name:10s}: {r['n_selected']} item, recall={r['bug_recall']:.1%}, APFD={r['apfd']:.1%}")

df_kernel = pd.DataFrame(kernel_results)
df_kernel.to_csv("results/real_exp2_kernel_sweep.csv", index=False)

plot_tc_bars(
    df_kernel["kernel"], df_kernel["TC_run"], df_kernel["bug_recall"],
    xlabel="Fungsi kemiripan (kernel)",
    title="Apakah pilihan fungsi kemiripan mengubah hasil? (data bug ASLI)",
    subtitle=f"vectorizer={best_method}; bar menunjukkan seberapa beda tiap kernel dalam jumlah item yang dibutuhkan",
    savepath="results/real_exp2_kernel_sweep.png",
    figsize=(9, 5),
    unit="item",
)
print(df_kernel.to_string(index=False))

df_kernel_sorted = df_kernel.sort_values(by=["bug_recall", "TC_run", "apfd"], ascending=[False, True, False])
BEST_KERNEL = df_kernel_sorted.iloc[0]["kernel"]
print(f"EXP2 winner: kernel='{BEST_KERNEL}'")


## EXP3 -- Apakah layak menyetel ulang hyperparameter kernel? (RQ3)

Length-scale tetap (1.0) vs disetel ulang lewat MLE (L-BFGS-B).

In [ ]:
best_method = BEST_METHOD
best_matrix = matrices[best_method]
HP_OPTIONS = ['fixed', 'mle']
print(f"EXP3: HP tuning sweep pakai vectorizer '{best_method}'")

hp_results = []
for hp_mode in HP_OPTIONS:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=groups, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=N_CASES, stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME, af_name=AF_NAME, hp_tuning=hp_mode, rng=np.random.default_rng(42)
    )
    hp_results.append({"hp_tuning": hp_mode, "TC_run": r["n_selected"], "bug_recall": r["bug_recall"],
                        "apfd": r["apfd"], "bugs_found": r["bugs_found"], "stopped_by_theta": r["stopped_by_theta"]})
    print(f"  hp_tuning={hp_mode:8s}: {r['n_selected']} item, recall={r['bug_recall']:.1%}, APFD={r['apfd']:.1%}")

df_hp = pd.DataFrame(hp_results)
df_hp.to_csv("results/real_exp3_hp_tuning_sweep.csv", index=False)
print(df_hp.to_string(index=False))

df_hp_sorted = df_hp.sort_values(by=["bug_recall", "TC_run", "apfd"], ascending=[False, True, False])
BEST_HP = df_hp_sorted.iloc[0]["hp_tuning"]
print(f"EXP3 winner: hp_tuning='{BEST_HP}'")

name_map_hp = {"fixed": "Manual (setelan tetap)", "mle": "Auto-tuning (disetel tiap langkah)"}
plot_tc_bars(
    [name_map_hp[h] for h in df_hp["hp_tuning"]], df_hp["TC_run"], df_hp["bug_recall"],
    xlabel="Strategi hyperparameter",
    title="Apakah layak membiarkan model menyetel dirinya sendiri? (data bug ASLI)",
    subtitle="Perbandingan jumlah item yang dibutuhkan sebelum berhenti",
    winner_idx=list(df_hp["hp_tuning"]).index(BEST_HP),
    savepath="results/real_exp3_hp_tuning_sweep.png",
    figsize=(8, 5),
    unit="item",
)


## EXP4 -- Aturan mana untuk memilih item berikutnya paling ampuh? (RQ4)

Cost-UCB, EI, LogEI, UCB, PI, Thompson Sampling.

In [ ]:
best_method = BEST_METHOD
best_matrix = matrices[best_method]
AF_OPTIONS = ['cost_ucb', 'ei', 'logei', 'ucb', 'pi', 'ts']
print(f"EXP4: Acquisition function sweep pakai vectorizer '{best_method}'")

af_results = []
for af_opt in AF_OPTIONS:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=groups, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=N_CASES, stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME, af_name=af_opt, hp_tuning=HP_TUNING, rng=np.random.default_rng(42)
    )
    af_results.append({"acquisition_fn": af_opt, "TC_run": r["n_selected"], "bug_recall": r["bug_recall"],
                        "apfd": r["apfd"], "bugs_found": r["bugs_found"], "stopped_by_theta": r["stopped_by_theta"]})
    print(f"  af={af_opt:10s}: {r['n_selected']} item, recall={r['bug_recall']:.1%}, APFD={r['apfd']:.1%}")

df_af = pd.DataFrame(af_results)
df_af.to_csv("results/real_exp4_af_sweep.csv", index=False)

name_map_af = {"cost_ucb": "Hemat biaya\n(default lama)", "ei": "Expected\nImprovement",
               "logei": "Improvement\n(versi stabil)", "ucb": "Berbasis\nkeyakinan",
               "pi": "Peluang\nImprovement", "ts": "Sampling\nacak"}
plot_tc_bars(
    [name_map_af[a] for a in df_af["acquisition_fn"]], df_af["TC_run"], df_af["bug_recall"],
    xlabel="Aturan memilih item berikutnya",
    title="Strategi mana yang temukan lebih banyak bug? (data bug ASLI)",
    subtitle="Default lama (hemat biaya) berhenti lebih awal dan melewatkan bug; beberapa lain menemukan semuanya",
    savepath="results/real_exp4_af_sweep.png",
    figsize=(10, 5.5),
    unit="item",
)
print(df_af.to_string(index=False))

df_af_sorted = df_af.sort_values(by=["bug_recall", "TC_run", "apfd"], ascending=[False, True, False])
BEST_AF = df_af_sorted.iloc[0]["acquisition_fn"]
print(f"EXP4 winner: acquisition_fn='{BEST_AF}'")


## EXP5 -- Seberapa besar UCB harus mendorong eksplorasi? (RQ5)

Sweep beta {1.0, 2.0, 3.0, 4.0}.

In [ ]:
best_method = BEST_METHOD
best_matrix = matrices[best_method]
BETA_OPTIONS = [1.0, 2.0, 3.0, 4.0]
print(f"EXP5: Beta sweep pakai vectorizer '{best_method}'")

beta_results = []
for b_val in BETA_OPTIONS:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=groups, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=N_CASES, stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME, af_name="ucb", hp_tuning=HP_TUNING, beta=b_val, rng=np.random.default_rng(42)
    )
    beta_results.append({"beta": b_val, "TC_run": r["n_selected"], "bug_recall": r["bug_recall"],
                          "apfd": r["apfd"], "bugs_found": r["bugs_found"], "stopped_by_theta": r["stopped_by_theta"]})
    print(f"  beta={b_val:.1f}: {r['n_selected']} item, recall={r['bug_recall']:.1%}, APFD={r['apfd']:.1%}")

df_beta = pd.DataFrame(beta_results)
df_beta.to_csv("results/real_exp5_beta_sweep.csv", index=False)
print(df_beta.to_string(index=False))

df_beta_sorted = df_beta.sort_values(by=["bug_recall", "TC_run", "apfd"], ascending=[False, True, False])
BEST_BETA = float(df_beta_sorted.iloc[0]["beta"])
print(f"EXP5 winner: beta={BEST_BETA}")

_beta_labels = [f"{b:.1f}" for b in df_beta["beta"]]
plot_pct_sweep(
    _beta_labels,
    {
        "Bug ditemukan (%)": (df_beta["bug_recall"], _ORANGE),
        f"Item dibaca (% dari {N_CASES})": (df_beta["TC_run"] / N_CASES, _BLUE),
    },
    xlabel="Level eksplorasi (beta)",
    title="Apakah mendorong eksplorasi mengubah hasil? (data bug ASLI)",
    subtitle=f"Beta {BEST_BETA:.1f} tetap temukan bug terbanyak; nilai lebih tinggi bisa terlalu percaya diri dan berhenti lebih cepat",
    chosen_label=f"{BEST_BETA:.1f}",
    savepath="results/real_exp5_beta_sweep.png",
)


## EXP6 -- Seberapa sensitif titik berhenti terhadap theta? (RQ6)

Tetap di Cost-UCB (bukan EI), sweep theta {0.50, 0.52, 0.53, 0.55} -- **grid yang sama persis dengan notebook utama**, sengaja tidak disetel ulang untuk skala 250-item, supaya terlihat apakah bentuk transisi tajamnya (kalau ada) konsisten di data nyata.

In [ ]:
THETA_VALUES = [0.50, 0.52, 0.53, 0.55]
best_method = BEST_METHOD
best_matrix = matrices[best_method]
print(f"EXP6: theta sweep pakai vectorizer '{best_method}' (Cost-UCB, dikunci)")

theta_results = []
for theta in THETA_VALUES:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=groups, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=N_CASES, stop_theta=theta,
        kernel_name=KERNEL_NAME, af_name="cost_ucb", rng=np.random.default_rng(42)
    )
    theta_results.append({"theta": theta, "TC_run": r["n_selected"], "bug_recall": r["bug_recall"],
                           "bugs_found": r["bugs_found"], "stopped_by_theta": r["stopped_by_theta"]})
    print(f"  theta={theta}: {r['n_selected']} item, recall={r['bug_recall']:.1%}")

df_theta = pd.DataFrame(theta_results)
df_theta.to_csv("results/real_exp6_theta_sweep.csv", index=False)

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(df_theta["theta"], df_theta["TC_run"], "o-b", label="Item Dibaca")
ax1.set_xlabel("Stopping Threshold (theta)")
ax1.set_ylabel("Item Dibaca", color="b")
ax2 = ax1.twinx()
ax2.plot(df_theta["theta"], df_theta["bug_recall"], "s--r", label="Bug Recall")
ax2.set_ylabel("Bug Recall", color="r")
ax2.set_ylim(0, 1.1)
ax1.set_title(f"EXP6: Theta Sweep, data bug ASLI (vectorizer={best_method})")
plt.tight_layout()
plt.savefig("results/real_exp6_theta_sweep.png", dpi=100)
plt.show()
print(df_theta.to_string(index=False))


## EXP7 -- Seberapa jauh ini lebih baik dari sekadar membaca acak? (RQ7)

Budget tetap 20/30/40/50 vs rata-rata 20 seed acak -- **pengecekan paling penting di notebook ini**.

In [ ]:
best_method = BEST_METHOD
best_matrix = matrices[best_method]
BUDGET_VALUES = [20, 30, 40, 50]
print(f"EXP7: Budget sweep pakai vectorizer '{best_method}'")

budget_results = []
for max_tc in BUDGET_VALUES:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=groups, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=max_tc, stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME, af_name=AF_NAME, rng=np.random.default_rng(42)
    )
    budget_results.append({"max_tc": max_tc, "TC_run": r["n_selected"], "bug_recall": r["bug_recall"],
                            "apfd": r["apfd"], "bugs_found": r["bugs_found"]})
    print(f"  max_tc={max_tc}: dibaca {r['n_selected']} item, recall={r['bug_recall']:.1%}, APFD={r['apfd']:.1%}")

df_budget = pd.DataFrame(budget_results)
df_budget.to_csv("results/real_exp7_budget_sweep.csv", index=False)

random_recalls = []
for max_tc in BUDGET_VALUES:
    recalls_seed = []
    for seed in range(20):
        rng_s = np.random.default_rng(seed)
        idx = rng_s.choice(N_CASES, size=max_tc, replace=False)
        recalls_seed.append(oracle[idx].sum() / oracle.sum())
    random_recalls.append(np.mean(recalls_seed))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_budget["max_tc"], df_budget["bug_recall"], "o-b", label="BO Reduction")
ax.plot(BUDGET_VALUES, random_recalls, "s--r", label="Random Baseline (rata-rata 20 seed)")
ax.axhline(1.0, color="gray", linestyle=":", label=f"Semua item ({N_CASES})")
ax.set_xlabel("Budget (maks item dibaca)")
ax.set_ylabel("Bug Recall")
ax.set_ylim(0, 1.1)
ax.set_title(f"EXP7: Budget Sweep, data bug ASLI (theta={STOP_THETA:.2f})")
ax.legend()
plt.tight_layout()
plt.savefig("results/real_exp7_budget_sweep.png", dpi=100)
plt.show()
print(df_budget.to_string(index=False))


## EXP8 -- Apakah hasil di atas runtuh tanpa korelasi konten? (RQ8)

Negative control yang SAMA PERSIS kodenya dengan notebook utama: oracle REAL sementara diganti 10 replikasi oracle acak-seragam (`draw_uniform_oracle`, {BUG_COUNT} bug ditaruh acak, tanpa korelasi ke teks). **Beda makna dari notebook utama:** di sana ini menguji "apakah keunggulan EXP1-7 cuma karena cara oracle DUMMY ditaruh". Di sini, karena EXP1-7 sudah pakai oracle REAL, EXP8 menguji pertanyaan berbeda: "kalau bug (real atau bukan) ditaruh TANPA korelasi ke teks sama sekali, apakah semua vectorizer collapse ke performa yang sama" -- sebagai pembanding, bukan sebagai audit atas oracle utama.

In [ ]:
def draw_uniform_oracle(case_count, bug_count, random_state):
    rng_f = np.random.default_rng(random_state)
    oracle_f = np.zeros(case_count)
    idx = rng_f.choice(case_count, size=bug_count, replace=False)
    oracle_f[idx] = 1.0
    return oracle_f

print("=== EXP8: FAIRNESS CHECK LEWAT 10 ORACLE SINTETIS ACAK-SERAGAM ===\n")
fairness_rows = []
for method_name, matrix in matrices.items():
    for rep in range(N_FAIRNESS_REPLICATES):
        oracle_f = draw_uniform_oracle(N_CASES, BUG_COUNT, rep)
        r = run_reduction_loop(
            matrix=matrix, ids=ids, menus=groups, costs=costs, oracle=oracle_f,
            initial_indices=initial_indices, max_tc=N_CASES, stop_theta=STOP_THETA,
            kernel_name=KERNEL_NAME, af_name=AF_NAME, rng=np.random.default_rng(rep)
        )
        fairness_rows.append({"method": method_name, "rep": rep, "n_selected": r["n_selected"],
                               "bug_recall": r["bug_recall"], "bugs_found": r["bugs_found"]})

df_fairness = pd.DataFrame(fairness_rows)
df_fair_summary = df_fairness.groupby("method").agg(
    mean_recall=("bug_recall", "mean"), std_recall=("bug_recall", "std"),
    mean_bugs=("bugs_found", "mean"), mean_tc=("n_selected", "mean")
).round(3)

print("=== EXP8 FAIRNESS SUMMARY (10 REPLIKASI PER METODE, DATA BUG ASLI) ===")
print(df_fair_summary.to_string())

os.makedirs("results", exist_ok=True)
df_fairness.to_csv("results/real_fairness_reduction.csv", index=False)
df_fair_summary.to_csv("results/real_fairness_reduction_summary.csv")
print("\nTersimpan real_fairness_reduction.csv dan real_fairness_reduction_summary.csv di results/")


## EXP9 -- Apakah keunggulan EXP1-EXP7 tahan terhadap kelompok mana yang "berisiko"? (RQ9)

Reuse `draw_clustered_oracle` yang sama persis dengan notebook utama, tapi kelompoknya `Priority` (bukan `Menu`). **Ingat keterbatasannya** (lihat markdown intro): cuma 5 nilai Priority, jadi "2 dari sekian kelompok berisiko" di sini mencakup proporsi kelompok jauh lebih besar (40%) dibanding notebook utama.

In [ ]:
group_arr = np.array(groups)
print("=== EXP9: UJI KETAHANAN ORACLE TERKELOMPOK (CONTENT-CORRELATED), KELOMPOK=Priority ===\n")
clustered_rows = []
for method_name, matrix in matrices.items():
    for rep in range(N_CLUSTERED_REPLICATES):
        oracle_c, risky_groups = draw_clustered_oracle(
            group_arr, case_count=N_CASES, bug_count=BUG_COUNT,
            n_risky_groups=N_RISKY_GROUPS, random_state=rep
        )
        r = run_reduction_loop(
            matrix=matrix, ids=ids, menus=groups, costs=costs, oracle=oracle_c,
            initial_indices=initial_indices, max_tc=N_CASES, stop_theta=STOP_THETA,
            kernel_name=KERNEL_NAME, af_name=AF_NAME, rng=np.random.default_rng(rep)
        )
        clustered_rows.append({"method": method_name, "rep": rep,
                                "risky_groups": ", ".join(map(str, risky_groups)),
                                "n_selected": r["n_selected"], "bug_recall": r["bug_recall"],
                                "bugs_found": r["bugs_found"]})

df_clustered = pd.DataFrame(clustered_rows)
df_clustered_summary = df_clustered.groupby("method").agg(
    mean_recall=("bug_recall", "mean"), std_recall=("bug_recall", "std"),
    mean_bugs=("bugs_found", "mean"), mean_tc=("n_selected", "mean"),
).round(3)

print("=== EXP9 CLUSTERED-ORACLE SUMMARY (10 REPLIKASI, Priority BERISIKO BEDA TIAP KALI) ===")
print(df_clustered_summary.to_string())

print(f"\nPembanding, EXP1 pakai oracle REAL tetap:")
for m, r in reduction_results.items():
    print(f"  {m:32s}: recall={r['bug_recall']:.1%}")

print("\nDan EXP8's oracle acak-seragam (content-independent):")
print(df_fair_summary.to_string())

os.makedirs("results", exist_ok=True)
df_clustered.to_csv("results/real_clustered_oracle_reduction.csv", index=False)
df_clustered_summary.to_csv("results/real_clustered_oracle_reduction_summary.csv")
print("\nTersimpan real_clustered_oracle_reduction.csv dan _summary.csv di results/")


### EXP9b -- Budget sweep, dirata-rata lintas pilihan kelompok berisiko berbeda

In [ ]:
best_method = BEST_METHOD
best_matrix = matrices[best_method]
BUDGET_VALUES = [20, 30, 40, 50]
print(f"EXP9b: Budget sweep dirata-rata lintas {N_CLUSTERED_REPLICATES} draw oracle terkelompok, vectorizer='{best_method}'")

clustered_budget_rows = []
for max_tc in BUDGET_VALUES:
    recalls_this_budget = []
    random_recalls_this_budget = []
    for rep in range(N_CLUSTERED_REPLICATES):
        oracle_rep, risky_groups_rep = draw_clustered_oracle(
            group_arr, case_count=N_CASES, bug_count=BUG_COUNT,
            n_risky_groups=N_RISKY_GROUPS, random_state=rep
        )
        r = run_reduction_loop(
            matrix=best_matrix, ids=ids, menus=groups, costs=costs, oracle=oracle_rep,
            initial_indices=initial_indices, max_tc=max_tc, stop_theta=STOP_THETA,
            kernel_name=KERNEL_NAME, af_name=AF_NAME, rng=np.random.default_rng(rep)
        )
        recalls_this_budget.append(r["bug_recall"])

        seed_recalls = []
        for seed in range(20):
            rng_s = np.random.default_rng(seed)
            idx = rng_s.choice(N_CASES, size=max_tc, replace=False)
            seed_recalls.append(oracle_rep[idx].sum() / oracle_rep.sum())
        random_recalls_this_budget.append(np.mean(seed_recalls))

    clustered_budget_rows.append({
        "max_tc": max_tc, "bo_mean_recall": np.mean(recalls_this_budget),
        "bo_std_recall": np.std(recalls_this_budget), "random_mean_recall": np.mean(random_recalls_this_budget),
    })
    print(f"  max_tc={max_tc}: BO mean recall={np.mean(recalls_this_budget):.1%} "
          f"(+/-{np.std(recalls_this_budget):.1%}), random mean={np.mean(random_recalls_this_budget):.1%}")

df_cbudget = pd.DataFrame(clustered_budget_rows)
df_cbudget.to_csv("results/real_clustered_oracle_budget_sweep.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(df_cbudget["max_tc"], df_cbudget["bo_mean_recall"], yerr=df_cbudget["bo_std_recall"],
            fmt="o-b", capsize=4, label=f"BO Reduction (rata-rata {N_CLUSTERED_REPLICATES} oracle)")
ax.plot(df_cbudget["max_tc"], df_cbudget["random_mean_recall"], "s--r", label="Random Baseline (20-seed avg)")
ax.axhline(1.0, color="gray", linestyle=":", label=f"Semua item ({N_CASES})")
ax.set_xlabel("Budget (maks item dibaca)")
ax.set_ylabel("Bug Recall")
ax.set_ylim(0, 1.1)
ax.set_title(f"EXP9b: Budget Sweep, data bug ASLI, dirata-rata {N_CLUSTERED_REPLICATES} oracle terkelompok")
ax.legend()
plt.tight_layout()
plt.savefig("results/real_clustered_oracle_budget_sweep.png", dpi=100)
plt.show()
print(df_cbudget.to_string(index=False))
print("\nEXP9 + EXP9b tersimpan di results/")


## EXP10 -- Gabungkan semua pemenang jadi satu pipeline BO utuh, lawan baseline acak (RQ10)

Sama persis dengan notebook utama: gabungkan pemenang EXP1-EXP5 + theta terkalibrasi jadi satu pipeline, bandingkan recall kumulatif vs baseline acak item-per-item (bukan cuma checkpoint budget seperti EXP7), plus visualisasi 3 langkah pertama (surrogate model + confidence band + skor acquisition).

In [ ]:
from sklearn.decomposition import PCA

print("=== EXP10: KONFIGURASI GABUNGAN (PEMENANG TIAP KOMPONEN), DATA BUG ASLI ===")
print(f"  vectorizer (EXP1) : {BEST_METHOD}")
print(f"  kernel     (EXP2) : {BEST_KERNEL}")
print(f"  hp_tuning  (EXP3) : {BEST_HP}")
print(f"  af_name    (EXP4) : {BEST_AF}")
print(f"  beta       (EXP5) : {BEST_BETA}")
print(f"  stop_theta (Step4): {STOP_THETA}")

final_matrix = matrices[BEST_METHOD]
final_result = run_reduction_loop(
    matrix=final_matrix, ids=ids, menus=groups, costs=costs, oracle=oracle,
    initial_indices=initial_indices, max_tc=N_CASES, stop_theta=STOP_THETA,
    kernel_name=BEST_KERNEL, af_name=BEST_AF, hp_tuning=BEST_HP, beta=BEST_BETA,
    rng=np.random.default_rng(42)
)
print(f"\nHasil gabungan: {final_result['n_selected']} item dibaca, "
      f"recall={final_result['bug_recall']:.1%}, APFD={final_result['apfd']:.1%}, "
      f"berhenti oleh theta={final_result['stopped_by_theta']}")

N_RANDOM_SEEDS = 20
warm_recall = oracle[initial_indices].sum() / oracle.sum()
bo_x = [len(initial_indices)] + [h["step"] for h in final_result["history"]]
bo_y = [warm_recall] + [h["bug_recall"] for h in final_result["history"]]

plot_upto = N_CASES
random_curves = []
for seed in range(N_RANDOM_SEEDS):
    rng_r = np.random.default_rng(seed)
    perm = rng_r.permutation(N_CASES)
    cum_recall = np.cumsum(oracle[perm]) / oracle.sum()
    random_curves.append(cum_recall[:plot_upto])
random_mean = np.mean(random_curves, axis=0)
random_x = np.arange(1, plot_upto + 1)

_reach = np.where(random_mean >= final_result["bug_recall"] - 1e-9)[0]
random_tc_to_match = int(random_x[_reach[0]]) if len(_reach) else None

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(bo_x, [y * 100 for y in bo_y], "o-", color=_BLUE, linewidth=2.5, markersize=5,
        label=f"Pipeline gabungan ({BEST_METHOD}, {BEST_KERNEL}, {BEST_AF})", zorder=3)
ax.plot(random_x, random_mean * 100, "s--", color=_ORANGE, linewidth=2, markersize=4,
        label=f"Baseline acak (rata-rata {N_RANDOM_SEEDS} seed)", zorder=3)
ax.axhline(100, color=_GRAY, linestyle=":", linewidth=1.5, zorder=1, label="Semua bug ditemukan")
ax.axvline(final_result["n_selected"], color=_BLUE_DARK, linestyle=(0, (4, 3)), linewidth=1.6, zorder=1)
ax.text(final_result["n_selected"], 104, f"{final_result['n_selected']} item\npilihan terbaik", ha="center", va="bottom",
        fontsize=9.5, color=_BLUE_DARK, fontweight="bold")
ax.set_xlabel("Bug report dibaca", fontsize=11, color=_SUB_C)
ax.set_ylabel("Bug ditemukan (kumulatif %)", fontsize=11, color=_SUB_C)
ax.set_ylim(0, 112)
ax.set_yticks([0, 20, 40, 60, 80, 100]); ax.set_yticklabels([f"{v}%" for v in [0, 20, 40, 60, 80, 100]])
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(_AXIS_C)
ax.tick_params(colors=_AXIS_C)
ax.yaxis.grid(True, color="#E7E6E1", linewidth=1)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, fontsize=9.5, labelcolor=_SUB_C)
fig.suptitle("Data bug ASLI: apakah gabungan semua pemenang benar-benar kalahkan baca acak?",
             x=0.01, ha="left", fontsize=14, fontweight="bold", color=_TITLE_C, y=1.0)
ax.set_title("Pipeline gabungan: pemenang EXP1-EXP5 dijalankan bersama, vs baseline acak rata-rata 20 seed",
             loc="left", fontsize=10, color=_SUB_C, pad=10)
plt.tight_layout(rect=[0, 0, 1, 0.90])
os.makedirs("results", exist_ok=True)
plt.savefig("results/real_exp10_final_combined_vs_random.png", dpi=150, bbox_inches="tight")
plt.show()

if random_tc_to_match:
    print(f"\nBO gabungan mencapai recall {final_result['bug_recall']:.1%} di {final_result['n_selected']} item. "
          f"Baseline acak baru mencapai recall setara itu di sekitar {random_tc_to_match} item "
          f"({random_tc_to_match - final_result['n_selected']:+d} dibanding BO).")
else:
    print(f"\nBaseline acak tidak pernah mencapai recall {final_result['bug_recall']:.1%} "
          f"dalam {plot_upto} item yang dicoba.")

pd.DataFrame({"tc": random_x, "random_mean_recall": random_mean}).to_csv(
    "results/real_exp10_random_baseline_curve.csv", index=False)
pd.DataFrame(final_result["history"]).to_csv("results/real_exp10_final_combined_history.csv", index=False)


# -- Visualisasi proses 3 langkah pertama, pakai konfigurasi gabungan --
N_DEMO_STEPS = 3
_pca_x = PCA(n_components=1, random_state=42).fit_transform(final_matrix).ravel()
_demo_selected = list(initial_indices)
_demo_rng = np.random.default_rng(42)
_snapshots = []

for _step in range(N_DEMO_STEPS):
    _candidates = [i for i in range(len(ids)) if i not in set(_demo_selected)]
    _train_x = final_matrix[_demo_selected]
    _train_y = np.array([oracle[i] for i in _demo_selected])
    _cand_x = final_matrix[_candidates]
    _cand_costs = np.array([costs[i] for i in _candidates])

    _mean, _std = _gp_predict(_train_x, _train_y, _cand_x,
                               sigma2=SIGMA2, noise=NOISE_ALPHA,
                               kernel_name=BEST_KERNEL, hp_tuning=BEST_HP)
    # FIX (dibanding notebook utama): notebook utama masih pakai mean(train_y)
    # di sini, padahal run_reduction_loop yang sesungguhnya sudah diperbaiki
    # jadi max(train_y). Cuma memengaruhi gambar demo, bukan angka final
    # manapun -- tapi tetap inkonsistensi matematis nyata, jadi diperbaiki.
    _best_so_far = float(np.max(_train_y)) if len(_train_y) > 0 else 0.0
    _scores = get_af_score(BEST_AF, _mean, _std, _cand_costs, _best_so_far, _demo_rng, beta=BEST_BETA)
    _chosen_local = int(np.argmax(_scores))
    _chosen = _candidates[_chosen_local]

    _snapshots.append({"selected_before": list(_demo_selected), "candidates": list(_candidates),
                        "mean": _mean, "std": _std, "scores": _scores, "chosen": _chosen})
    _demo_selected.append(_chosen)

fig, axes = plt.subplots(N_DEMO_STEPS, 2, figsize=(12, 3.2 * N_DEMO_STEPS))

for _row, _snap in enumerate(_snapshots):
    ax_top, ax_bot = axes[_row, 0], axes[_row, 1]

    _order = np.argsort(_pca_x[_snap["candidates"]])
    _cx = _pca_x[_snap["candidates"]][_order]
    _cm = _snap["mean"][_order]
    _cs = _snap["std"][_order]
    ax_top.plot(_cx, _cm, color="steelblue", lw=1.6, label="Prediksi model (mean_prob)")
    ax_top.fill_between(_cx, np.clip(_cm - _cs, 0, 1), np.clip(_cm + _cs, 0, 1),
                         color="steelblue", alpha=0.25, label="Ketidakyakinan (\u00b1std)")

    _sel = _snap["selected_before"]
    _sel_x = _pca_x[_sel]
    _sel_y = oracle[_sel]
    ax_top.scatter(_sel_x[_sel_y == 1], [1.0] * int((_sel_y == 1).sum()),
                    color="firebrick", marker="o", s=45, zorder=5, label="Bug report dibaca, bug ditemukan")
    ax_top.scatter(_sel_x[_sel_y == 0], [0.0] * int((_sel_y == 0).sum()),
                    color="black", marker="o", s=30, zorder=5, label="Bug report dibaca, bukan bug serius")

    _chosen_i = _snap["chosen"]
    _chosen_y = float(_snap["mean"][_snap["candidates"].index(_chosen_i)])
    ax_top.scatter([_pca_x[_chosen_i]], [_chosen_y], color="red", marker="*", s=220,
                    zorder=6, label="Baru dipilih langkah ini")

    ax_top.set_ylim(-0.05, 1.05)
    ax_top.set_ylabel("P(bug serius+fixed)")
    ax_top.set_title(f"Langkah {_row + 1}: surrogate model gabungan ({len(_sel)} item sudah dibaca)")
    if _row == 0:
        ax_top.legend(fontsize=7, loc="upper right")

    _cscore = _snap["scores"][_order]
    ax_bot.plot(_cx, _cscore, color="seagreen", lw=1.4)
    ax_bot.fill_between(_cx, 0, _cscore, color="seagreen", alpha=0.15)
    _max_local = int(np.argmax(_snap["scores"]))
    ax_bot.scatter([_pca_x[_snap["candidates"][_max_local]]],
                    [_snap["scores"][_max_local]], color="red", marker="v", s=110, zorder=6)
    ax_bot.set_title(f"Langkah {_row + 1}: skor acquisition function ({BEST_AF}), \u25bc = terpilih")
    ax_bot.set_ylabel("Skor")
    if _row == N_DEMO_STEPS - 1:
        ax_top.set_xlabel("Posisi bug report (proyeksi PCA-1 dari vektor teks)")
        ax_bot.set_xlabel("Posisi bug report (proyeksi PCA-1 dari vektor teks)")

plt.tight_layout()
plt.savefig("results/real_exp10_process_visualization.png", dpi=100)
plt.show()

print(f"\nBug report terpilih di 3 langkah pertama run gabungan ini: {[ids[s['chosen']] for s in _snapshots]}")
print("\nSemua file EXP10 tersimpan di results/ dengan prefix real_")


## Cara membaca semua ini

Tiap bagian EXP di atas menjawab satu pertanyaan sempit, dengan semua yang lain dikunci ke default. Tidak satupun -- sendiri atau bersama -- membuktikan setelan mana yang "terbaik" secara universal; semuanya menunjukkan bagaimana model *berperilaku* pada satu skenario NYATA (bukan sintetis) tapi tetap dengan sejumlah pilihan desain (proxy biaya, kelompok Priority sebagai analog Menu, oversampling proporsi positif).

EXP1-EXP7 dan EXP10 di sini berjalan langsung di atas oracle bug REAL (bukan disuntikkan) -- ini beda mendasar dari notebook utama. EXP8 adalah kontrol negatif: mengulang semuanya dengan penempatan bug acak-seragam sintetis (independen dari konten, 10 replikasi) untuk mengecek keunggulan di atas bukan cuma kebisingan. EXP9 menguji ketahanan ke arah lain: 10 replikasi dengan pilihan kelompok Priority berisiko yang berbeda-beda (dengan keterbatasan cuma-5-kelompok yang sudah diakui). EXP10 adalah sintesis: pemenang aktual EXP1-EXP5 dijalankan sekali sebagai satu pipeline, head-to-head lawan baseline acak.

**Bandingkan dengan `Uji Coba Reduction.ipynb`:** kalau bentuk hasil di sini (vectorizer lexical menang, keunggulan BO vs acak tipis atau tidak) konsisten dengan yang sudah ditemukan di sana pasca-fix `best_so_far`, itu bukti independen yang memperkuat temuan itu. Kalau berbeda jauh, itu tanda pola di notebook utama spesifik ke cara oracle dummy-nya dibuat -- keduanya harus dilaporkan apa adanya ke sensei, bukan dipilih salah satu.


## Membungkus semuanya

Semua CSV/JSON/PNG di atas dibungkus jadi satu zip untuk arsip/serah-terima, persis seperti notebook utama.

In [ ]:
import zipfile
from pathlib import Path

results_dir = Path("results")
zip_path = Path("reduction_experiment_results_realdata_full.zip")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in results_dir.glob('real_*'):
        if file.is_file():
            zipf.write(file, arcname=file.name)
            print(f"Zipped: {file.name}")

print(f"\nSemua log EXP0-EXP10 (data bug ASLI) dibungkus ke: {zip_path.resolve()}")
